In [1]:
from dataclasses import dataclass, field

import cv2
import numpy as np
from glow import around
from tqdm.auto import tqdm

N = 1024
HSIZE = 256
SPEED = 1

ACCEL = 1.5  # 20% accel on collision
FORKS = 4  # 4x more particles on collision

RNG = np.random.default_rng(42)

NC = 256
COLOR = RNG.random((NC, 3), dtype='f4')
COLOR *= 255 / (COLOR.sum(1, keepdims=True) + 1e-7)
COLOR = around(COLOR.clip(0, 255), dtype='u1')
COLOR[0] = 0


@dataclass
class State:
    # (n 2 2), f
    box: np.ndarray = field(default_factory=lambda: np.empty((0, 2, 2), 'f4'))
    # (n 2), f
    vec: np.ndarray = field(default_factory=lambda: np.empty((0, 2), 'f4'))
    # (n), f
    speed: np.ndarray = field(default_factory=lambda: np.empty(0, 'f4'))
    # (n), u1
    color_id: np.ndarray = field(default_factory=lambda: np.empty(0, 'u1'))
    # (n), f
    hsize: np.ndarray = field(default_factory=lambda: np.empty(0, 'f4'))

    def __len__(self):
        return len(self.hsize)

    def extend(self, speed: np.ndarray, hsize: np.ndarray):
        n = len(speed)
        pos = np.full((n, 2), N/2, dtype='f4')
        box = np.stack([pos - hsize[:, None], pos + hsize[:, None]], axis=1)  # (n lo-hi x-y)

        angle = RNG.random(n, dtype='f4')
        angle *= 2 * np.pi
        vec = np.c_[speed * np.cos(angle), speed * np.sin(angle)]

        color_id = RNG.integers(1, NC, size=n, dtype='u1')

        self.box = np.r_[self.box, box]
        self.vec = np.r_[self.vec, vec]
        self.speed = np.r_[self.speed, speed]
        self.color_id = np.r_[self.color_id, color_id]
        self.hsize = np.r_[self.hsize, hsize]

    def update(self):
        n = len(s)

        self.box += self.vec[:, None, :]
        oob = (self.box[:, 1, :].max(1) >= N) | (self.box[:, 0, :].min(1) < 0)
        if not oob.any():
            return

        ib = ~oob
        self.box, _ = self.box[ib], self.box[oob]
        self.vec, vec = self.vec[ib], self.vec[oob]
        self.speed, speed = self.speed[ib], self.speed[oob]
        self.color_id, _ = self.color_id[ib], self.color_id[oob]
        self.hsize, hsize = self.hsize[ib], self.hsize[oob]

        speed *= ACCEL
        hsize /= np.sqrt(FORKS)
        self.extend(
            speed=np.repeat(speed, FORKS),
            hsize=np.repeat(hsize, FORKS),
        )

    def render(self, tq2):
        box = around(self.box, dtype='i2')

        if 1:
            im = np.zeros((N, N), dtype='u1')
            for ((x1, y1), (x2, y2)), c in zip(box, self.color_id):
                im[y1:y2, x1:x2] = c
                # cv2.rectangle(im, (x1, y1), (x2, y2), c.tolist(), -1)
                tq2.update()
            im = COLOR[im]

        else:
            color = COLOR[self.color_id]
            ctcs = [{}, {}, {}]
            for ((x1, y1), (x2, y2)), c in zip(box, color.tolist()):
                for tcs, t in zip(ctcs, c):
                    tcs.setdefault(t, []).append(np.array([[x1, y1], [x1, y2], [x2, y2], [x2, y1]], dtype='i4'))

            im = np.zeros((3, N, N), dtype='u1')
            for im_, tcs in zip(im, ctcs):
                for t, cs in tcs.items():
                    cv2.drawContours(im_, cs, -1, t, -1)
                    tq2.update(len(cs))

            im = im.transpose(1, 2, 0)

        cv2.imshow('win', im)


s = State()
s.extend(
    speed=np.full(3, SPEED, dtype='f4'),
    hsize=np.full(3, HSIZE, dtype='f4'),
)
try:
    with tqdm(desc='frames') as tq:
        with tqdm(desc='draw calls') as tq2:
            while (cv2.waitKey(10) & 0xFF) != ord('q'):
                s.update()
                s.render(tq2)
                tq.update()
                tq.set_postfix(num_particles=len(s))
finally:
    cv2.destroyAllWindows()


frames: 0it [00:00, ?it/s]

draw calls: 0it [00:00, ?it/s]